In [ ]:
import torch, os, sys
sys.path.append('../')
from torch import nn, Tensor
import numpy as np
from transformers import AutoTokenizer
from model_loader import LlamaModelDownload
from data_loader_llama3 import get_dataloader
from config_llama3 import get_config

/home/new/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model_name: str = r"Qwen3-1.7B"
config = get_config()

In [3]:
cuda_ = torch.device('cuda' if torch.cuda.is_available else 'cpu')
llmdl = LlamaModelDownload(model_name, device=cuda_)
llmdl.start(True)

model in local


`torch_dtype` is deprecated! Use `dtype` instead!


{0: '22GB', 'cpu': '30GB'}


Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.17s/it]

The Attention path of model /home/new/Documents/LLModel/Qwen3-1.7B is: 2048


(Qwen3Model(
   (embed_tokens): Embedding(151670, 2048)
   (layers): ModuleList(
     (0-27): 28 x Qwen3DecoderLayer(
       (self_attn): Qwen3Attention(
         (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
         (k_proj): Linear4bit(in_features=2048, out_features=1024, bias=False)
         (v_proj): Linear4bit(in_features=2048, out_features=1024, bias=False)
         (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
         (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
         (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
       )
       (mlp): Qwen3MLP(
         (gate_proj): Linear4bit(in_features=2048, out_features=6144, bias=False)
         (up_proj): Linear4bit(in_features=2048, out_features=6144, bias=False)
         (down_proj): Linear4bit(in_features=6144, out_features=2048, bias=False)
         (act_fn): SiLUActivation()
       )
       (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
       (post_attention_layernorm): Qwen3RMSNorm

In [4]:
model_path: str = os.path.join(r"../../LLModel", model_name)

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    use_fast=True,  # Llama3建议使用slow tokenizer
    trust_remote_code=True,
    fix_mistral_regex=True
)

train, val, test = get_dataloader(tokenizer, config)

加载数据: ./data/emotion_cls/processed_data.pkl
Overall data size: 51239 样本
Semi-supervised采样: 492 labeled blocks (20.0%)

数据集划分:
  Train: 492 blocks
  Val:   197 blocks
  Test:  197 blocks
  Train prompts: 10317
  Val prompts:   4247
  Test prompts:  3922



In [12]:
for i, obj in enumerate(train):
    if i>2: break
    target_words = obj['input_ids']
    for ii, words in enumerate(target_words):
        # tokens = tokenizer.convert_ids_to_tokens(words)
        sentence = tokenizer.decode(words, skip_special_tokens=True)
        print(f"[batch{i}, {ii}th sentence with emotion [{obj['emotion_text'][ii]}]] get result of {sentence}")

[batch0, 0th sentence with emotion [faithful]] get result of <|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an emotion recognition assistant. Analyze the dialogue and classify the emotion of the user's final response.<|eot_id|><|start_header_id|>user<|end_header_id|>

History: 
Situation: i have always been loyal to my wife .
Context: [Asker: ] [User: i have never cheated on my wife .]<|eot_id|><|start_header_id|>assistant<|end_header_id|>


[batch0, 1th sentence with emotion [faithful]] get result of <|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an emotion recognition assistant. Analyze the dialogue and classify the emotion of the user's final response.<|eot_id|><|start_header_id|>user<|end_header_id|>

History: i have never cheated on my wife .
Situation: i have always been loyal to my wife .
Context: [Asker: and thats something you should never do , good on you .] [User: yea it has n't been easy but i am proud i have n't]<|eot_id|><|start

In [10]:
obj['emotion_text']

['sad',
 'disappointed',
 'disappointed',
 'angry',
 'angry',
 'hopeful',
 'hopeful',
 'hopeful']

In [1]:
import re

target = """
==================================================
Final Results Summary
==================================================

Test Metrics:
  accuracy: 0.6249
  f1_macro: 0.6237
  f1_weighted: 0.6207
  precision: 0.6553
  recall: 0.6303

Best Validation Metrics:
  val_accuracy: 0.6376
  val_f1_macro: 0.6180
  val_f1_weighted: 0.6228
  val_precision: 0.6462
  val_recall: 0.6296

==================================================
"""

In [22]:
pattern = r"(\w+):\s?(\d*+\.\d*)"
match = re.findall(pattern, target)

In [1]:
from predict_label_output import get_core_intel

/home/new/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path_dir = r"./outputs/Llama3_01-28/lr_3e-05_bs_4_ws_3"
result = get_core_intel(path_dir)

✓ Parsed summary.txt: ./outputs/Llama3_01-28/lr_3e-05_bs_4_ws_3/summary.txt
✓ Parsed training.log: ./outputs/Llama3_01-28/lr_3e-05_bs_4_ws_3/training.log


In [3]:
result

{'test_accuracy': 0.6249,
 'test_f1_macro': 0.6237,
 'test_f1_weighted': 0.6207,
 'test_precision': 0.6553,
 'test_recall': 0.6303,
 'val_accuracy': 0.6376,
 'val_f1_macro': 0.618,
 'val_f1_weighted': 0.6228,
 'val_precision': 0.6462,
 'val_recall': 0.6296,
 'train_sample': 10605,
 'val_sample': 4252,
 'test_sample': 3879}

In [1]:
from predict_label_output import get_best_checkpoint, find_experiment_dirs, batch_inference_all_experiments

/home/new/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
exp_dirs = find_experiment_dirs()
print(*exp_dirs, sep="\n")

Found 15 completed experiments
./outputs/Llama3_01-28/lr_3e-05_bs_4_ws_3
./outputs/Qwen3_1d7_02-05/lr_2e-05_bs_16_ws_3_Qwen1d7_SSP03
./outputs/Qwen3_1d7_02-05/lr_2e-05_bs_16_ws_3_Qwen1d7_SSP01
./outputs/Qwen3_1d7_02-05/lr_3e-05_bs_16_ws_3_Qwen1d7_FSL8
./outputs/Llama3_01-30/lr_2e-05_bs_4_ws_3_wo_prompt_SP
./outputs/Llama3_01-30/lr_2e-05_bs_4_ws_3
./outputs/Llama3_01-30/lr_3e-05_bs_4_ws_3
./outputs/Llama3_01-27/lr_2e-05_bs_4_ws_3
./outputs/Llama3_01-27/lr_3e-05_bs_4_ws_3
./outputs/Qwen3_02-06/method_FSL_bs_16_inputdata_ws_prompt_Qwen1d7_FSL16
./outputs/Qwen3_02-06/method_SSP_bs_6_inputdata_ws_prompt_Qwen4b_SSP03
./outputs/Qwen3_02-06/method_SSP_bs_6_inputdata_ws_prompt_Qwen4b_SSP01
./outputs/Llama3_01-31/lr_2e-05_bs_4_ws_3_full_train811
./outputs/Llama3_01-31/lr_2e-05_bs_4_ws_3_SSP_01_full
./outputs/Llama3_01-31/lr_3e-05_bs_4_ws_3_FSL_4_full


In [3]:
print(get_best_checkpoint(exp_dirs[1]))

4000


In [2]:
batch_inference_all_experiments()

Found 15 completed experiments

Processing 15 experiments on cuda:3
Dataset: Validation


[Llama3_01-28/lr_3e-05_bs_4_ws_3]
✓ Parsed summary.txt: ./outputs/Llama3_01-28/lr_3e-05_bs_4_ws_3/summary.txt
✓ Parsed training.log: ./outputs/Llama3_01-28/lr_3e-05_bs_4_ws_3/training.log
./outputs/Llama3_01-28/lr_3e-05_bs_4_ws_3/checkpoints/checkpoint-3000
{'test_accuracy': 0.6249, 'test_f1_macro': 0.6237, 'test_f1_weighted': 0.6207, 'test_precision': 0.6553, 'test_recall': 0.6303, 'val_accuracy': 0.6376, 'val_f1_macro': 0.618, 'val_f1_weighted': 0.6228, 'val_precision': 0.6462, 'val_recall': 0.6296, 'train_sample': 10605, 'val_sample': 4252, 'test_sample': 3879}
./outputs/Llama3_01-28/lr_3e-05_bs_4_ws_3/predictions_step3000_val.txt

[Qwen3_1d7_02-05/lr_2e-05_bs_16_ws_3_Qwen1d7_SSP03]
✓ Parsed summary.txt: ./outputs/Qwen3_1d7_02-05/lr_2e-05_bs_16_ws_3_Qwen1d7_SSP03/summary.txt
✓ Parsed training.log: ./outputs/Qwen3_1d7_02-05/lr_2e-05_bs_16_ws_3_Qwen1d7_SSP03/training.log
./outputs/Qwen3_1d7_02-05

(0, '')